# Compute Subreddit Similarity — BucketedRandomProjectionLSH

Input: `subreddit_title_embeddings_distilbert.parquet`  
Output: `subreddit_cosine_similarity_lsh.parquet` — cùng format với brute-force output  
Môi trường: PySpark local mode (`local[*]`)  
Yêu cầu: Java JDK 8 hoặc 11, `pyspark` package.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.ml.feature import BucketedRandomProjectionLSH, Normalizer
from pyspark.ml.linalg import Vectors
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

# Chạy từ notebook/ hoặc processing-local/ đều được.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

INPUT_PATH = REPO_ROOT / "data/processed/subreddit_title_embeddings_distilbert.parquet"
OUTPUT_PATH = REPO_ROOT / "data/processed/subreddit_cosine_similarity_lsh.parquet"

BUCKET_LENGTH = 0.2
NUM_HASH_TABLES = 3
# cosine >= ~0.955 → Euclidean <= sqrt(2*(1-0.955)) ≈ 0.3
# Dùng rộng hơn mức threshold thực tế để không bỏ sót ứng viên.
EUCLIDEAN_THRESHOLD = 0.3
SIMILARITY_EDGE_PERCENTILE = 97.0

assert INPUT_PATH.exists(), f"Missing: {INPUT_PATH}"
INPUT_PATH

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SubredditLSH")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
# Chuyển pandas embedding sang Spark DenseVector.
df_pd = pd.read_parquet(INPUT_PATH)
data = [
    (row["subreddit"], Vectors.dense(row["embedding"]))
    for _, row in df_pd.iterrows()
]
df_spark = spark.createDataFrame(data, ["subreddit", "features"])

# L2-normalize: với unit vector, cosine similarity = dot product.
normalizer = Normalizer(inputCol="features", outputCol="norm_features", p=2.0)
df_norm = normalizer.transform(df_spark)

brp = BucketedRandomProjectionLSH(
    inputCol="norm_features",
    outputCol="hashes",
    bucketLength=BUCKET_LENGTH,
    numHashTables=NUM_HASH_TABLES,
)
model = brp.fit(df_norm)

print(f"Subreddits loaded: {df_spark.count()}")
print("LSH model fitted.")

In [ ]:
# Self-join tìm các cặp gần nhau. Lọc A < B để loại self-pair và duplicate.
df_candidates = (
    model
    .approxSimilarityJoin(df_norm, df_norm, EUCLIDEAN_THRESHOLD, distCol="euclidean_dist")
    .filter(F.col("datasetA.subreddit") < F.col("datasetB.subreddit"))
)
print(f"Candidate pairs: {df_candidates.count():,}")


@F.udf(FloatType())
def dot_udf(v1, v2):
    # Dot product = cosine similarity cho L2-normalized vectors.
    return float(np.dot(np.array(v1), np.array(v2)))


df_with_sim = df_candidates.select(
    F.col("datasetA.subreddit").alias("source_subreddit"),
    F.col("datasetB.subreddit").alias("target_subreddit"),
    dot_udf(
        F.col("datasetA.norm_features"),
        F.col("datasetB.norm_features"),
    ).alias("cosine_similarity"),
)

# Cùng phương pháp lọc với brute-force: giữ top 3% (percentile 97).
threshold = df_with_sim.approxQuantile(
    "cosine_similarity", [SIMILARITY_EDGE_PERCENTILE / 100], 0.001
)[0]

df_filtered = (
    df_with_sim
    .filter(F.col("cosine_similarity") >= threshold)
    .orderBy(F.col("cosine_similarity").desc())
)

edge_frame = df_filtered.toPandas()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
edge_frame.to_parquet(OUTPUT_PATH, index=False)

print(f"Similarity threshold (P{SIMILARITY_EDGE_PERCENTILE:.0f}): {threshold:.10f}")
print(f"Strong edges: {len(edge_frame):,}")
print(f"Saved to: {OUTPUT_PATH}")
edge_frame.head(10)